# 01 — Ingestão e Bronze (exercício)

**Objetivo:** praticar ingestão via HuggingFace, normalização de JSON e escrita de Parquet.

Execute os notebooks em ordem.

## Onde estamos no pipeline

![Star schema alvo](../docs/images/image.png)

Este é o estado final que vamos atingir ao terminar a sessão: uma fato `fact_report` com chaves estrangeiras para quatro dimensões. Cada notebook contribui com uma camada do caminho até esse esquema.

Posição deste notebook (camada **bronze**):

> Fonte → **Bronze** → Staging → Dim/Fact → Checks → Profiling → Marts/Views → Dashboard

Aqui você baixa o dataset do HuggingFace, normaliza campos aninhados em `*_json` e grava `data/bronze_hackerone_reports.parquet`. Esse arquivo é a entrada da próxima etapa.

In [ ]:
# Parametros usados pelo Papermill/Airflow.
run_date = None
project_root = None

from pathlib import Path
import sys

for candidato in [Path.cwd(), *Path.cwd().parents]:
    caminhos_common = [
        candidato / "_common.py",
        candidato / "aula02" / "_common.py",
        candidato / "exercicios" / "_common.py",
        candidato / "sessao-02-data-architecture" / "_common.py",
        candidato / "sessao-02-data-architecture" / "exercicios" / "_common.py",
    ]
    encontrado = next((p for p in caminhos_common if p.exists()), None)
    if encontrado:
        sys.path.insert(0, str(encontrado.parent))
        break
else:
    raise FileNotFoundError("Nao encontrei _common.py a partir do diretorio atual.")

from _common import configurar_paths, conectar_duckdb
paths = configurar_paths(project_root)
globals().update(paths)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Banco DuckDB: {DB_PATH}")


## Exercício
Complete os TODOs para carregar os três splits, normalizar campos aninhados e gravar a camada bronze.

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import json

HUGGINGFACE_DATASET = "Hacker0x01/disclosed_reports"

splits = ["train", "test", "validation"]
frames = [load_dataset(HUGGINGFACE_DATASET, split=split).to_pandas() for split in splits]
df = pd.concat(frames, ignore_index=True)
print(f"Linhas carregadas: {len(df):,}")
df.head(3)


In [ ]:
def para_json(valor):
    if valor is None:
        return None
    if isinstance(valor, float) and np.isnan(valor):
        return None
    if isinstance(valor, str):
        return valor if valor.strip() else None
    return json.dumps(valor, ensure_ascii=False, default=str)

json_cols = ["reporter_json", "team_json", "weakness_json", "structured_scope_json"]
for col in json_cols:
    if col in df.columns:
        df[col] = df[col].map(para_json)

bronze_cols = [
    "id", "title", "created_at", "disclosed_at", "substate", "visibility", "has_bounty?",
    "vote_count", "original_report_id", "reporter_json", "team_json", "weakness_json",
    "structured_scope_json", "vulnerability_information",
]
bronze = df[[c for c in bronze_cols if c in df.columns]].copy()
bronze.to_parquet(BRONZE_PARQUET, index=False)
bronze.to_csv(RAW_CSV, index=False)

print(f"Bronze Parquet gravado em: {BRONZE_PARQUET}")
print(f"CSV bruto gravado em: {RAW_CSV}")
print(f"Shape bronze: {bronze.shape}")
bronze.head(3)
